# CodeAlpha Task 2 — Emotion Recognition from Speech

Use MFCC speech features and a CNN to classify emotions. The CodeAlpha brief specifies MFCCs, CNN/RNN/LSTM approaches, and RAVDESS/TESS/EMO-DB as suitable datasets.

## RAVDESS dataset

Official RAVDESS source: https://zenodo.org/records/1188976

A 16-kHz version is available at: https://zenodo.org/records/11063852

RAVDESS contains speech audio with eight emotion labels encoded in the filename. The full audio dataset is not redistributed here.

In [ ]:
# pip install librosa tensorflow scikit-learn
import os, glob
import numpy as np
import librosa
import tensorflow as tf
from tensorflow.keras import layers, models

DATA_DIR='Audio_Speech_Actors_01-24'

emotion_map={'01':'neutral','02':'calm','03':'happy','04':'sad',
             '05':'angry','06':'fearful','07':'disgust','08':'surprised'}

def emotion_from_filename(path):
    return emotion_map[os.path.basename(path).split('-')[2]]

def extract_mfcc(path):
    audio,sr=librosa.load(path,sr=16000,duration=3)
    return librosa.feature.mfcc(y=audio,sr=sr,n_mfcc=40)

In [ ]:
# Full RAVDESS training pipeline
wavs=glob.glob(os.path.join(DATA_DIR,'Actor_*','*.wav'))
X=np.array([extract_mfcc(w) for w in wavs])[...,None]
labels=np.array([emotion_from_filename(w) for w in wavs])
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
enc=LabelEncoder(); y=enc.fit_transform(labels)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)

model=models.Sequential([
    layers.Input(shape=X_train.shape[1:]),
    layers.Conv2D(32,3,activation='relu',padding='same'),
    layers.MaxPooling2D(),
    layers.Conv2D(64,3,activation='relu',padding='same'),
    layers.MaxPooling2D(),
    layers.Conv2D(128,3,activation='relu'),
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation='relu'),
    layers.Dropout(.3),
    layers.Dense(len(enc.classes_),activation='softmax')
])
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
model.fit(X_train,y_train,validation_split=.15,epochs=25,batch_size=32)
print(model.evaluate(X_test,y_test))

## Offline demo

`demo_mfcc_features.csv` is a small synthetic MFCC-shaped dataset included only so the package can be inspected without downloading RAVDESS. It is **not** a RAVDESS benchmark and its metrics must not be presented as real speech-emotion performance.